# 01 — Panel construction

Reproduces the data pipeline described in **Methods: Data source and sample construction**
of *"Deepening before dissolution: non-monotonic escape dynamics in the scaling exponents
of county economies"*.

This notebook (i) lists the registry of raw CSMAR workbooks that feed the pipeline,
(ii) verifies the cleaned panels shipped in `data_clean/`, and (iii) reports the panel
dimensions, per-year county counts, Fokker-Planck period definitions, and key-variable
coverage quoted in the paper. Only aggregates (shapes, counts, coverage fractions) are
printed — never raw county-level values.

In [1]:
import os, sys
sys.path.insert(0, '/home/yushentang/EconP-release')
os.chdir('/home/yushentang/EconP-release')

import pandas as pd
from src import config

## 1. Raw data registry

The panel is built from 22 county-level Excel workbooks from the CSMAR China county
database. **The raw workbooks are not shipped with this repository** (CSMAR licensing);
acquisition instructions are given in `DATA_SOURCES.md`. Twenty workbooks enter the
pipeline through the registry in `src/config.py` (`THEMED_FILES`, `LONG_FORMAT_FILES`,
`POVERTY_FILE`, `MAIN_DATABASE`); the remaining two belong to the same CSMAR bundle
but are not used by any analysis in the paper. Names below are the original CSMAR
filenames with an English gloss.

In [2]:
# Registry as declared in src/config.py (names only -- the files themselves are not shipped)
themed = sorted({v["file"] for v in config.THEMED_FILES.values()}
                | {v["file"] for v in config.LONG_FORMAT_FILES.values()})
pipeline_files = themed + [config.POVERTY_FILE["file"], config.MAIN_DATABASE["file"]]

# Two workbooks from the same CSMAR bundle, not referenced by the pipeline
aux_files = ["县域城镇单位从业人员与工资.xlsx",
             "县域对外贸易.xlsx"]

gloss = {
    "地区生产总值及指数.xlsx": "Regional GDP and GDP indices",
    "人口.xlsx": "Population",
    "农业产值.xlsx": "Agricultural gross output value",
    "土地面积.xlsx": "Land area (administrative, cultivated)",
    "农业生产条件.xlsx": "Agricultural production conditions",
    "城乡居民收入.xlsx": "Urban and rural household income",
    "金融机构存贷款.xlsx": "Financial institution deposits and loans",
    "县域从业人员统计.xlsx": "County employment statistics",
    "县域地理空间联系.xlsx": "County geospatial coordinates",
    "县域固定资产投资.xlsx": "County fixed-asset investment",
    "县域社会消费品零售总额.xlsx": "County retail sales of consumer goods",
    "县域卫生条件.xlsx": "County health conditions",
    "社会保险与福利.xlsx": "Social insurance and welfare",
    "财政收支.xlsx": "Fiscal revenue and expenditure",
    "教育基本情况.xlsx": "Basic education statistics",
    "交通运输与邮电.xlsx": "Transport, post and telecommunications",
    "农作物播种面积.xlsx": "Crop sown area (total and grain)",
    "农产品产量.xlsx": "Agricultural product output (grain)",
    "全国832个国家级贫困县名单、摘帽整理数据.xlsx": "National poverty-county list (832 counties, exit years)",
    "中国县域数据库6.0版_0514090132.xlsx": "China county database v6.0 (cross-validation)",
    "县域城镇单位从业人员与工资.xlsx": "Urban-unit employment and wages (bundle only, unused)",
    "县域对外贸易.xlsx": "County foreign trade (bundle only, unused)",
}

all_files = pipeline_files + aux_files
for i, f in enumerate(all_files, 1):
    tag = "pipeline" if f in pipeline_files else "unused  "
    print(f"{i:2d}. [{tag}] {f}  --  {gloss[f]}")
print(f"\nTotal workbooks in CSMAR bundle: {len(all_files)}   (paper reports: 22)")
print(f"Workbooks entering the pipeline registry: {len(pipeline_files)}")

 1. [pipeline] 交通运输与邮电.xlsx  --  Transport, post and telecommunications
 2. [pipeline] 人口.xlsx  --  Population
 3. [pipeline] 农业产值.xlsx  --  Agricultural gross output value
 4. [pipeline] 农业生产条件.xlsx  --  Agricultural production conditions
 5. [pipeline] 农产品产量.xlsx  --  Agricultural product output (grain)
 6. [pipeline] 农作物播种面积.xlsx  --  Crop sown area (total and grain)
 7. [pipeline] 县域从业人员统计.xlsx  --  County employment statistics
 8. [pipeline] 县域卫生条件.xlsx  --  County health conditions
 9. [pipeline] 县域固定资产投资.xlsx  --  County fixed-asset investment
10. [pipeline] 县域地理空间联系.xlsx  --  County geospatial coordinates
11. [pipeline] 县域社会消费品零售总额.xlsx  --  County retail sales of consumer goods
12. [pipeline] 土地面积.xlsx  --  Land area (administrative, cultivated)
13. [pipeline] 地区生产总值及指数.xlsx  --  Regional GDP and GDP indices
14. [pipeline] 城乡居民收入.xlsx  --  Urban and rural household income
15. [pipeline] 教育基本情况.xlsx  --  Basic education statistics
16. [pipeline] 社会保险与福利.xlsx  --  Social insuran

## 2. Guard: cleaned panels must be present

The pipeline output (`data_clean/panel_full.parquet`, `data_clean/panel_scaling.parquet`)
ships with the repository, so this cell should pass. If you deleted `data_clean/` you
must first re-acquire the raw workbooks (see `DATA_SOURCES.md`) and rebuild the panel
with the pipeline scripts in `src/` (`data_loader` → `data_cleaner` → `deflator` →
`data_merger`).

In [3]:
if not os.path.exists('data_clean/panel_full.parquet'):
    raise FileNotFoundError(
        "data_clean/panel_full.parquet not found.\n"
        "The cleaned panels ship with this repository; if data_clean/ was removed,\n"
        "re-acquire the raw CSMAR workbooks (instructions in DATA_SOURCES.md) and\n"
        "rebuild the panel with the src/ pipeline before running the notebooks."
    )
print("Guard passed: cleaned panels found in data_clean/.")

Guard passed: cleaned panels found in data_clean/.


## 3. Panel overview

Two parquet files are produced by the pipeline:

- `panel_full.parquet` — every county-year observation after cleaning (coverage and
  audit checks);
- `panel_scaling.parquet` — the analysis panel, restricted to counties with the
  variables required by the scaling regressions. All scaling, Fokker-Planck, and
  early-warning results in the paper use this panel.

In [4]:
panel_full = pd.read_parquet('data_clean/panel_full.parquet')
panel_scaling = pd.read_parquet('data_clean/panel_scaling.parquet')

print(f"panel_full:    {panel_full.shape[0]:,} rows x {panel_full.shape[1]} cols, "
      f"{panel_full['CountyCode'].nunique():,} counties")
print("               paper reports: 38,308 rows x 64 cols, 2,297 counties")
print(f"panel_scaling: {panel_scaling.shape[0]:,} rows x {panel_scaling.shape[1]} cols, "
      f"{panel_scaling['CountyCode'].nunique():,} counties")
print("               paper reports: 19,392 rows, 1,882 counties")
print(f"Year range:    {panel_full['Year'].min()}-{panel_full['Year'].max()}"
      "   (paper reports: 2000-2023)")

panel_full:    38,308 rows x 64 cols, 2,297 counties
               paper reports: 38,308 rows x 64 cols, 2,297 counties
panel_scaling: 19,392 rows x 64 cols, 1,882 counties
               paper reports: 19,392 rows, 1,882 counties
Year range:    2000-2023   (paper reports: 2000-2023)


## 4. Fokker-Planck period definitions

The escape-dynamics analysis splits the panel into three policy periods around the
targeted poverty-alleviation campaign. The definitions live in
`src/config.py::FP_PERIODS` so that every downstream script uses the same split.

In [5]:
for name, (y0, y1) in config.FP_PERIODS.items():
    print(f"{name:>6s}: {y0}-{y1}")
print("paper reports: pre 2000-2012, during 2013-2019, post 2020-2023")

   pre: 2000-2012
during: 2013-2019
  post: 2020-2023
paper reports: pre 2000-2012, during 2013-2019, post 2020-2023


## 5. Per-year county counts

County coverage is not balanced: CSMAR reporting expands through the 2000s and thins
in the post-2020 years (the attrition that makes the post-period Kramers-Moyal
coefficients unidentifiable, as discussed in the paper).

In [6]:
for name, df in [("panel_full", panel_full), ("panel_scaling", panel_scaling)]:
    per_year = df.groupby('Year')['CountyCode'].nunique()
    print(f"{name}: counties per year  min={per_year.min()}, "
          f"median={per_year.median():.1f}, max={per_year.max()}")


panel_full: counties per year  min=634, median=1638.5, max=1788
panel_scaling: counties per year  min=137, median=862.5, max=981


## 6. Key-variable coverage

Coverage (share of rows with a non-missing value) for the two variables whose
missingness shapes the analysis sample: `CultivateArea` (limits the multi-factor
scaling regressions) and `GrainOutput`.

In [7]:
for var, paper_line in [("CultivateArea", "~45% (full) to ~53% (analysis panel)"),
                        ("GrainOutput", "85.9% (full panel)")]:
    cov_full = 100 * panel_full[var].notna().mean()
    cov_scal = 100 * panel_scaling[var].notna().mean()
    print(f"{var:14s}: full {cov_full:.1f}%, scaling {cov_scal:.1f}%   "
          f"(paper reports: {paper_line})")

CultivateArea : full 44.6%, scaling 53.0%   (paper reports: ~45% (full) to ~53% (analysis panel))
GrainOutput   : full 85.9%, scaling 88.6%   (paper reports: 85.9% (full panel))


## Summary

The shipped parquets reproduce the sample described in the Methods: a full panel of
38,308 county-year rows (64 variables, 2,297 counties, 2000-2023) and an analysis
panel of 19,392 rows (1,882 counties), with the three Fokker-Planck periods defined
in `src/config.py`. No raw county-level values are displayed in this notebook.